# 01 — Data ExtractionLoad raw NYC taxi data and inspect structure, missing values, duplicates, and date ranges.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

## Project Paths

In [2]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
docs_dir = project_root / "docs"
reports_dir = project_root / "reports"

processed_data_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

trip_data_path = raw_data_dir / "yellow_tripdata_2024-01.parquet"
zone_lookup_path = raw_data_dir / "taxi_zone_lookup.csv"

print(f"Project root: {project_root}")
print(f"Trip data path exists: {trip_data_path.exists()}")
print(f"Zone lookup path exists: {zone_lookup_path.exists()}")

Project root: /Users/angelonelson/Projects/NYCTaxiTripAnalytics
Trip data path exists: True
Zone lookup path exists: True


## Validate Raw Files

In [3]:
required_files = {
    "Yellow Taxi Trip Records January 2024": trip_data_path,
    "Taxi Zone Lookup Table": zone_lookup_path,
}

missing_files = []

for file_description, file_path in required_files.items():
    if not file_path.exists():
        missing_files.append((file_description, file_path))

if missing_files:
    for file_description, file_path in missing_files:
        print(f"Missing file: {file_description}")
        print(f"Expected location: {file_path}")
    raise FileNotFoundError("One or more required raw files are missing. Download them before continuing.")

print("All required raw files are available.")

All required raw files are available.


## Load Raw Data

In [4]:
taxi_trips_raw = pd.read_parquet(trip_data_path)
taxi_zones_raw = pd.read_csv(zone_lookup_path)

print("Raw taxi trips loaded successfully.")
print(f"Taxi trips shape: {taxi_trips_raw.shape[0]:,} rows and {taxi_trips_raw.shape[1]:,} columns")

print("\nTaxi zone lookup loaded successfully.")
print(f"Zone lookup shape: {taxi_zones_raw.shape[0]:,} rows and {taxi_zones_raw.shape[1]:,} columns")

Raw taxi trips loaded successfully.
Taxi trips shape: 2,964,624 rows and 19 columns

Taxi zone lookup loaded successfully.
Zone lookup shape: 265 rows and 4 columns


## Preview Trip Data

In [5]:
taxi_trips_raw.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.00,1.72,1.00,N,186,79,2,17.70,1.00,0.50,0.00,0.00,1.00,22.70,2.50,0.00
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.00,1.80,1.00,N,140,236,1,10.00,3.50,0.50,3.75,0.00,1.00,18.75,2.50,0.00
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.00,4.70,1.00,N,236,79,1,23.30,3.50,0.50,3.00,0.00,1.00,31.30,2.50,0.00
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.00,1.40,1.00,N,79,211,1,10.00,3.50,0.50,2.00,0.00,1.00,17.00,2.50,0.00
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.00,0.80,1.00,N,211,148,1,7.90,3.50,0.50,3.20,0.00,1.00,16.10,2.50,0.00


## Preview Zone Lookup

In [6]:
taxi_zones_raw.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


## Column Summary

In [7]:
raw_trip_column_summary = pd.DataFrame({
    "column_name": taxi_trips_raw.columns,
    "data_type": taxi_trips_raw.dtypes.astype(str).values,
    "non_null_count": taxi_trips_raw.notna().sum().values,
    "missing_count": taxi_trips_raw.isna().sum().values,
    "missing_percentage": (taxi_trips_raw.isna().mean() * 100).round(2).values,
})

raw_trip_column_summary

,column_name,data_type,non_null_count,missing_count,missing_percentage
0,VendorID,int32,2964624,0,0.00
1,tpep_pickup_datetime,datetime64[us],2964624,0,0.00
2,tpep_dropoff_datetime,datetime64[us],2964624,0,0.00
3,passenger_count,float64,2824462,140162,4.73
4,trip_distance,float64,2964624,0,0.00
5,RatecodeID,float64,2824462,140162,4.73
6,store_and_fwd_flag,str,2824462,140162,4.73
7,PULocationID,int32,2964624,0,0.00
8,DOLocationID,int32,2964624,0,0.00
9,payment_type,int64,2964624,0,0.00


In [8]:
zone_column_summary = pd.DataFrame({
    "column_name": taxi_zones_raw.columns,
    "data_type": taxi_zones_raw.dtypes.astype(str).values,
    "non_null_count": taxi_zones_raw.notna().sum().values,
    "missing_count": taxi_zones_raw.isna().sum().values,
    "missing_percentage": (taxi_zones_raw.isna().mean() * 100).round(2).values,
})

zone_column_summary

,column_name,data_type,non_null_count,missing_count,missing_percentage
0,LocationID,int64,265,0,0.00
1,Borough,str,264,1,0.38
2,Zone,str,264,1,0.38
3,service_zone,str,263,2,0.75


## Missing Values

In [9]:
missing_value_summary = (
    taxi_trips_raw
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column_name", 0: "missing_count"})
)

missing_value_summary["missing_percentage"] = (
    missing_value_summary["missing_count"] / len(taxi_trips_raw) * 100
).round(2)

missing_value_summary = missing_value_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

missing_value_summary

,column_name,missing_count,missing_percentage
18,Airport_fee,140162,4.73
17,congestion_surcharge,140162,4.73
3,passenger_count,140162,4.73
5,RatecodeID,140162,4.73
6,store_and_fwd_flag,140162,4.73
11,extra,0,0.00
16,total_amount,0,0.00
15,improvement_surcharge,0,0.00
14,tolls_amount,0,0.00
13,tip_amount,0,0.00


## Duplicate Check

In [10]:
duplicate_row_count = taxi_trips_raw.duplicated().sum()
duplicate_row_percentage = duplicate_row_count / len(taxi_trips_raw) * 100

print(f"Duplicate rows: {duplicate_row_count:,}")
print(f"Duplicate row percentage: {duplicate_row_percentage:.4f}%")

Duplicate rows: 0
Duplicate row percentage: 0.0000%


## Numeric Summary

In [11]:
numeric_columns = taxi_trips_raw.select_dtypes(include=["number"]).columns.tolist()

numeric_summary = taxi_trips_raw[numeric_columns].describe().T
numeric_summary["missing_count"] = taxi_trips_raw[numeric_columns].isna().sum()
numeric_summary["missing_percentage"] = (taxi_trips_raw[numeric_columns].isna().mean() * 100).round(2)

numeric_summary

,count,mean,std,min,25%,50%,75%,max,missing_count,missing_percentage
VendorID,"2,964,624.00",1.75,0.43,1.00,2.00,2.00,2.00,6.00,0,0.00
passenger_count,"2,824,462.00",1.34,0.85,0.00,1.00,1.00,1.00,9.00,140162,4.73
trip_distance,"2,964,624.00",3.65,225.46,0.00,1.00,1.68,3.11,"312,722.30",0,0.00
RatecodeID,"2,824,462.00",2.07,9.82,1.00,1.00,1.00,1.00,99.00,140162,4.73
PULocationID,"2,964,624.00",166.02,63.62,1.00,132.00,162.00,234.00,265.00,0,0.00
DOLocationID,"2,964,624.00",165.12,69.32,1.00,114.00,162.00,234.00,265.00,0,0.00
payment_type,"2,964,624.00",1.16,0.58,0.00,1.00,1.00,1.00,4.00,0,0.00
fare_amount,"2,964,624.00",18.18,18.95,-899.00,8.60,12.80,20.50,"5,000.00",0,0.00
extra,"2,964,624.00",1.45,1.80,-7.50,0.00,1.00,2.50,14.25,0,0.00
mta_tax,"2,964,624.00",0.48,0.12,-0.50,0.50,0.50,0.50,4.00,0,0.00


## Date Range Check

In [12]:
pickup_column = "tpep_pickup_datetime"
dropoff_column = "tpep_dropoff_datetime"

if pickup_column in taxi_trips_raw.columns and dropoff_column in taxi_trips_raw.columns:
    print("Pickup datetime range:")
    print(f"Minimum pickup datetime: {taxi_trips_raw[pickup_column].min()}")
    print(f"Maximum pickup datetime: {taxi_trips_raw[pickup_column].max()}")

    print("\nDropoff datetime range:")
    print(f"Minimum dropoff datetime: {taxi_trips_raw[dropoff_column].min()}")
    print(f"Maximum dropoff datetime: {taxi_trips_raw[dropoff_column].max()}")
else:
    print("Expected pickup/dropoff datetime columns were not found.")

Pickup datetime range:
Minimum pickup datetime: 2002-12-31 22:59:39
Maximum pickup datetime: 2024-02-01 00:01:15

Dropoff datetime range:
Minimum dropoff datetime: 2002-12-31 23:05:41
Maximum dropoff datetime: 2024-02-02 13:56:52


## Categorical Value Counts

In [13]:
categorical_columns_to_check = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
]

for column_name in categorical_columns_to_check:
    if column_name in taxi_trips_raw.columns:
        print(f"\nValue counts for {column_name}:")
        print(taxi_trips_raw[column_name].value_counts(dropna=False).head(20))


Value counts for VendorID:
VendorID
2    2234632
1     729732
6        260
Name: count, dtype: int64

Value counts for RatecodeID:
RatecodeID
1.00     2663350
NaN       140162
2.00       98713
99.00      28663
5.00       19410
3.00        7954
4.00        6365
6.00           7
Name: count, dtype: int64

Value counts for store_and_fwd_flag:
store_and_fwd_flag
N      2813126
NaN     140162
Y        11336
Name: count, dtype: int64

Value counts for payment_type:
payment_type
1    2319046
2     439191
0     140162
4      46628
3      19597
Name: count, dtype: int64


## Export Profile Outputs

In [14]:
raw_trip_column_summary.to_csv(processed_data_dir / "raw_trip_column_summary.csv", index=False)
missing_value_summary.to_csv(processed_data_dir / "raw_missing_value_summary.csv", index=False)
numeric_summary.to_csv(processed_data_dir / "raw_numeric_summary.csv")

profile_markdown = f"""# Raw Dataset Profile

## Dataset

NYC Yellow Taxi Trip Records — January 2024

## Source Files

- yellow_tripdata_2024-01.parquet
- taxi_zone_lookup.csv

## Shape

- Taxi trip rows: {taxi_trips_raw.shape[0]:,}
- Taxi trip columns: {taxi_trips_raw.shape[1]:,}
- Taxi zone lookup rows: {taxi_zones_raw.shape[0]:,}
- Taxi zone lookup columns: {taxi_zones_raw.shape[1]:,}

## Duplicate Rows

- Duplicate taxi trip rows: {duplicate_row_count:,}
- Duplicate row percentage: {duplicate_row_percentage:.4f}%

## Pickup Date Range

- Minimum pickup datetime: {taxi_trips_raw[pickup_column].min() if pickup_column in taxi_trips_raw.columns else "Not available"}
- Maximum pickup datetime: {taxi_trips_raw[pickup_column].max() if pickup_column in taxi_trips_raw.columns else "Not available"}

## Dropoff Date Range

- Minimum dropoff datetime: {taxi_trips_raw[dropoff_column].min() if dropoff_column in taxi_trips_raw.columns else "Not available"}
- Maximum dropoff datetime: {taxi_trips_raw[dropoff_column].max() if dropoff_column in taxi_trips_raw.columns else "Not available"}

## Notes

This profile is generated before cleaning. The cleaning notebook will handle invalid dates, duplicate rows, missing values, impossible trip values, and feature engineering.
"""

with open(docs_dir / "raw_dataset_profile.md", "w", encoding="utf-8") as file:
    file.write(profile_markdown)

print("Raw dataset profiling outputs saved successfully.")

Raw dataset profiling outputs saved successfully.


## DoneExtraction complete. Next: `02_cleaning.ipynb`.